# Lecture 12 Attention & Transformer

## 一、 Image Captioning (图像描述生成)

这是一个典型的**多模态 (Multimodal) 任务**，核心架构是 **CNN (Encoder) + RNN (Decoder)**。

### 1. 特征提取与网络架构 (CNN端)
*   **特征维度的选择**：使用CNN（如VGGNet）来提取图像特征。
    *   不能太低维/不能用最终的Logits：因为Logits只包含高度浓缩的类别分布信息，丢失了图像的空间细节和丰富的语义属性。
    *   不能太高维（如早期的卷积层特征）：维度过高RNN难以处理且未经过充分的语义抽象。
    *   最终选择：通常截断最后的全连接分类层和Softmax（如图中的 FC-1000 ），而是**提取倒数第二层的特征向量（如 FC-4096）**作为整张图像的全局特征表示。
*   **训练策略**：通常会 **Freeze CNN (stop gradient)**。即不更新CNN的权重，只把它当作一个静态的特征提取器，集中算力训练后端的RNN模型。

### 2. 文本生成机制 (RNN端)
图像特征如何喂给RNN？
*   **图像特征注入**：CNN提取的特征向量经过一个线性映射层，转换成初始的偏置向量 $b_v$：
    $$b_v = W_{hi}[CNN_{\theta_c}(I)]$$
*   **修改后的RNN单步更新公式**：
    $$h_t = f(W_{hx}x_t + W_{hh}h_{t-1} + b_h + \mathbf{1}(t=1) \odot b_v)$$
    *注：这表明图像特征 $b_v$ 仅在第一步（$t=1$）作为隐状态 $h$ 的初始偏置被注入，后续RNN主要依赖自身的隐状态循环和之前生成的词汇。*
*   **自回归生成 (Autoregressive Generation)**：
    *   输入 `<START>` token ($x_0$) 触发生成。
    *   结合图像初始化的隐状态，生成第一个词（如 "straw"）。
    *   **Sample (采样)**：将生成的词作为下一步的输入 $x_1$，继续更新隐状态生成下一个词（如 "hat"）。
    *   直到模型输出特殊的 **`<END>` token**，生成过程终止（注意：`<END>` token 和 句号并不等同，句号还可以继续输出，END则是真的结束了）

### 3. 结果与局限性
*   **小数据集的退化现象**：当训练数据不足时，模型本质上只是在做 Caption Retrieval（检索），它学到的是输出训练集中与当前图片CNN特征最相近 (Nearest Neighbor) 的描述，而非真正的“看图说话”。
*   **Failure Cases (失败案例的局限性)**：这个结构理论上能描述所有图片，**但是缺乏真正的视觉和语义对应（Visual-Semantic Grounding）能力**。它很容易被图像的局部特征或训练集的偏置误导，产生幻觉 (Hallucination)：
    *   例如：把穿着毛皮大衣的女人误识别为“手里拿着一只猫”。
    *   例如：把倒立的腿的形状误识别为“拿着冲浪板站在沙滩上”。
    *   这说明简单的“CNN全局向量注入RNN”的方式，无法处理复杂的空间关系和细粒度对象。

---

## 二、 VQA (Visual Question Answering, 视觉问答)

相比于单向生成的Captioning，VQA要求模型根据**图像 (Image)** 和**自然语言问题 (Question)** 联合输出一个**答案 (Answer)**。

### 1. 双流特征对齐架构 (Feature Alignment)
核心思想：“image encode和language encode的feature进行对齐”。具体的Pipeline如下：
*   **Image Encode (视觉流)**：
    *   图片 $\rightarrow$ 经过 VGGNet 等CNN $\rightarrow$ 提取 4096维的全局特征 $\rightarrow$ 经过 Fully-Connected 层 $\rightarrow$ 映射为一个 **1024 维的图像特征向量**。
*   **Language Encode (语言流)**：
    *   问题 ("How many horses are in this image?") $\rightarrow$ 经过词嵌入 $\rightarrow$ 经过 LSTM 层（如 2x2x512 LSTM）对序列进行建模 $\rightarrow$ 取最后的隐状态经过 Fully-Connected 层 $\rightarrow$ 映射为一个 **1024 维的文本特征向量**。
*   **Feature Fusion (特征融合/对齐)**：
    *   将同维度的图像特征和文本特征进行 **逐元素相乘 (Point-wise multiplication)**。
    *   这种操作强制让图像和文本特征在同一个子空间中产生交互。
*   **Answer Prediction (预测分类)**：
    *   融合后的向量再经过 Fully-Connected 层 $\rightarrow$ Softmax 分类器，输出整个候选答案词表中的概率分布（例如分类输出数字 "2"）。

### 2. 延伸应用：Visual Dialog (视觉对话)
*   不再是单轮的一问一答，而是**基于图像进行多轮连续的对话 (Conversations about Images)**。
*   这要求模型不仅要理解图片特征和当前问题，还要引入对**上下文历史对话记录 (History)** 的记忆和处理。

## 一、 注意力机制的起源：从RNN到Attention

### 1. Seq2Seq模型与瓶颈问题
*   **传统Encoder-Decoder架构**：输入序列经过Encoder压缩成一个**固定长度的上下文向量 (Context Vector, $c$)**，Decoder根据该向量生成输出序列（如机器翻译）。
*   **核心瓶颈**：无论输入序列多长（如$T=1000$），都必须被压缩到一个固定大小的向量$c$中，导致长序列信息丢失（信息瓶颈）。
*   必考：训练过程中，不论上一步输出的正确与否，给下一步都固定输入正确结果，来训练。

### 2. 引入注意力机制 (Attention)
*   **核心思想**：Decoder在生成每一个词时，不再依赖单一的固定上下文向量，而是**动态地“回头看”**输入序列的所有部分，对相关的部分给予更多的“注意力”。
*   **计算步骤**：
    1.  **计算对齐分数 (Alignment Scores)**：衡量当前Decoder状态 $s_{t-1}$ 与各个Encoder隐状态 $h_i$ 的相关性。
        $$e_{t,i} = f_{att}(s_{t-1}, h_i)$$
    2.  **计算注意力权重 (Attention Weights)**：使用Softmax对分数进行归一化，得到权重 $a_{t,i} \in (0,1)$ 且和为1。
        $$a_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$$
    3.  **计算动态上下文向量 (Context Vector)**：将Encoder的隐状态按权重进行加权求和。
        $$c_t = \sum_i a_{t,i} h_i$$
    4.  **生成输出**：使用 $c_t$ 参与Decoder当前步的计算。

## 二、 注意力层的通用抽象：Q、K、V

注意力机制被推广为一种处理向量集合的通用网络操作。
*   **Query (Q)**：查询向量（我要找什么）。
*   **Key (K)**：键向量（输入信息的特征标签，用于和Q匹配）。
*   **Value (V)**：值向量（输入信息的实际内容，用于加权求和）。
*   **操作本质**：每一个 Query 会与所有的 Key 计算相似度，然后根据相似度对所有的 Value 进行线性组合（加权求和），得到该 Query 的输出。

---

## 三、 自注意力机制 (Self-Attention Layer)

### 1. 核心概念
*   **输入即全部**：Query、Key、Value全部来自**同一个输入序列** $X$。每个输入元素都在“观察”序列中的所有其他元素，从而聚合信息。
*   **线性投影**：输入矩阵 $X \in \mathbb{R}^{N \times D_{in}}$ 通过学习到的权重矩阵投影得到 Q、K、V：
    $$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

实际上 **Q、K、V 是完全并行的**。
在实际代码实现中，对于输入矩阵 $X$，我们不会分别去乘三次权重矩阵，而是把 $W_Q, W_K, W_V$ **拼接 (Concatenate) 成一个巨大的权重矩阵**。
$$[Q, K, V] = X \cdot [W_Q, W_K, W_V]$$
只需要**一次**矩阵乘法（One Matmul），就可以同时得到 Q、K、V。这也是 Transformer 在 GPU 上计算极快、高度并行的核心原因。


### 2. 缩放点积注意力 (Scaled Dot-Product Attention)
这是Transformer中最核心的公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{D_Q}}\right)V$$
**为什么要除以 $\sqrt{D_Q}$ (缩放因子)？**
**原理展开：**
假设 Query ($q$) 和 Key ($k$) 都是维度为 $D$ 的向量，且它们的元素是相互独立、均值为0、方差为1的随机变量。
当我们计算它们的点积时：
$$q \cdot k = \sum_{i=1}^D q_i k_i$$
由于包含了 $D$ 个元素的累加，这个**点积结果的方差会随着维度 $D$ 的增大而增大**（具体来说，方差约等于 $D$），那么点积的标准差就是 $\sqrt{D}$。这就意味着，当维度很大时（比如 $D=1024$），点积的结果中可能会出现绝对值极大的数（比如 $[10, 20, 30]$ 甚至更大）。

**为什么大数值对 Softmax 是致命的？**
Softmax 的公式是：$a_i = \frac{\exp(e_i)}{\sum_j \exp(e_j)}$
它对输入数值的大小**极端敏感**。如果输入是 $[1, 2, 3]$，Softmax输出可能是 $[0.09, 0.24, 0.67]$，梯度很好。但如果输入因为方差变大变成了 $[10, 20, 30]$，Softmax 的输出会变成近似 $[0, 0, 1]$。
*   **饱和与梯度消失**：当 Softmax 的输出变成这种接近 one-hot 的形式（即所谓的“饱和”），你在求导时会发现，$\frac{\partial \text{softmax}}{\partial e} \approx 0$。梯度几乎为0，也就是**梯度消失 (Gradient Vanishing)**，这会导致模型根本无法更新权重，训练完全停滞。
*   **解决方案**：强制把点积结果除以 $\sqrt{D}$（即 **Scaled** Dot-Product），强行把方差拉回 1，使得数值落入 Softmax 敏感且梯度健康的区间。

### 3. 自注意力的特性
*   **置换等变性 (Permutation Equivariant)**：自注意力机制本身是一个集合操作，它**不感知序列中元素的顺序**。如果打乱输入，输出同样会被打乱，但对应关系不变。
*   位置编码 (Positional Embedding) 的三大痛点与 RoPE ：
1.  **痛点1：说不清是 X 本身的信息，还是 Embedding 的信息**
    *   最初的 Transformer 是直接把位置向量和词向量**相加**：$X_{input} = X_{token} + P_{position}$。
    *   这确实很反直觉，相当于把“你是谁(词义)”和“你在哪(位置)”硬揉在一个向量里。模型必须靠自己强大的拟合能力，在后续的线性变换中再把它们解耦出来，这并不优雅。
2.  **痛点2：超出训练长度怎么外推 (Extrapolation)？**
    *   如果训练时只见过最大长度 $L=1024$，那位置 $1025$ 的 Embedding 是没有学过的，甚至是随机的。测试时遇到长文本模型直接崩溃。这叫**长度泛化 / 外推性极差**。
3.  **RoPE (旋转位置编码) 的原理与解决之道：**
    *   **原理**：RoPE 放弃了“绝对位置加法”，改用“绝对位置对应旋转角度”。想象 Q 和 K 是二维平面上的向量。如果 Q 在第 $m$ 个位置，我就把它旋转 $m\theta$ 度；K 在第 $n$ 个位置，我就把它旋转 $n\theta$ 度。
    *   **魔法所在**：当旋转后的 Q 和 K 做点积算相似度时，根据初中三角函数公式，**它们的点积结果完全只取决于它们的角度差，即 $(m-n)\theta$**。
    *   **为什么牛？** 
        1. 它**没有破坏原始词向量的加法空间**，而是施加了正交的旋转变换。
        2. 它在算相似度时，自动且完美地体现了**相对位置 (Relative Position)**（比如“猫”在“吃”后面隔了1个词，不管是在句首还是句尾，相对距离都是1）。相对位置比绝对位置对语言理解重要得多。

---

## 四、 注意力机制的变体

### 1. 掩码自注意力 (Masked Self-Attention)
*   **应用场景**：自回归语言模型（如预测下一个词）。
*   **机制**：在计算出相似度矩阵 $QK^T$ 后，将当前词“未来”位置的相似度强制设为 $-\infty$。经过 Softmax 后，这些位置的权重变为 0。
*   **目的**：防止模型在预测时“提前看到”未来的信息 (Look ahead)。

### 2. 多头自注意力 (Multi-headed Self-Attention)
*   **机制**：并行运行 $H$ 个独立的自注意力层（称为 Heads）。
*   **目的**：让模型在不同的表示子空间中捕捉不同类型的信息。
*   **输出融合**：将 $H$ 个头的输出拼接 (Concat)，再通过一个线性投影矩阵 $W_O$ 融合数据。
    $$\text{Outputs: } O = YW_O$$
*   **原理与结构**：
    *   假设我们的特征维度是 $D=512$。如果只做一次 Attention，模型只能学到一种关注模式。
    *   **多头 (Multi-head)** 就是把这个 $D$ 拆成 $H$ 份（比如 $H=8$ 头，每头维度 $D_H = 64$）。
    *   然后我们并行地跑 8 个各自独立的 Self-Attention。每个头都有自己专属的 $W_Q, W_K, W_V$ 权重。
    *   最后，把 8 个头的输出向量重新拼凑 (Concat) 在一起，过一个线性层 $W_O$ 融合，恢复成 512 维。
*   **作用**：
    *   **子空间特征提取**：每个头负责关注序列中**不同维度的语义关系**。比如，头1专门找“主谓关系”（"I" 关注 "love"）；头2专门找“指代关系”（"it" 关注 "apple"）；头3专门找情感词……
    *   这有点像 CNN 里面使用多个不同的**卷积核 (Filters)** 提取边缘、纹理等不同特征，多头注意力让模型在同一个词上捕捉到丰富的、多维度的上下文关联。

### 3. 计算复杂度与显存瓶颈
*   计算 $QK^T$ 会产生大小为 $N \times N$ 的注意力权重矩阵。
*   时间和空间复杂度均为 **$O(N^2)$**，这在处理长序列时极其昂贵。（注：FlashAttention 通过优化显存 I/O 缓解了这一问题）。

#### Attention 的四步矩阵乘法

**四个关键步骤：**
1.  **QKV Projection (投影)**: 计算 $Q, K, V$。（时间/空间复杂度约 $O(N)$）
2.  **QK Similarity (算相似度矩阵 $E$)**: 计算 $Q \times K^T$。（生成 $N \times N$ 矩阵）
3.  **V-Weighting (算加权和 $Y$)**: 计算 $\text{Softmax}(E) \times V$。（用到 $N \times N$ 矩阵）
4.  **Output Projection (输出投影)**: 融合结果输出 $Y$。

**瓶颈在哪里？**
致命瓶颈在**第 2 步和第 3 步**。如果你输入 $N=100K$ 个词（长文本），你要在 GPU 内存（HBM）里凭空实例化一个 $100,000 \times 100,000$ 的矩阵 $E$ 和注意力权重矩阵 $A$。如 PPT 108页所示，仅仅存这一个矩阵就需要 1.2 TB 的显存！GPU 直接 Out of Memory (OOM)。

#### FlashAttention
**FlashAttention 为什么能把 Memory 降到 $O(N)$？**
*   **传统做法的弊端**：算完第2步，必须把巨大无比的 $N \times N$ 矩阵**写入**慢速的 GPU 显存(HBM)，然后再为了第3步的 Softmax **读取**出来。这种频繁的 I/O 操作极慢且极度占内存。
*   **FlashAttention 的魔法 (Tiling & Recomputation)**：
    1.  **分块计算 (Tiling)**：它不一次性算整个 $N \times N$ 矩阵。而是把 $Q, K, V$ 切成小块 (Blocks)，每次只把一小块放进 GPU 内部极小但极快的 SRAM 缓存中。
    2.  **Kernel 融合 (Fusion)**：在超快的 SRAM 里，算出一小块的 $Q \times K^T$，**立刻就地算 Softmax（利用特殊的在线Softmax算法），立马乘上对应的 $V$ 块**，得到局部输出。
    3.  既然这块输出已经算完了，那个 $N \times N$ 里的对应局部小方块**就可以直接丢弃了，完全不需要写进 HBM**！
*   **结论**：它在数学上算出的结果和标准 Attention **一模一样 (Exact)**，但因为巧妙地避开了实例化 $N \times N$ 矩阵，使得**峰值内存消耗从 $O(N^2)$ 骤降到 $O(N)$**，同时因为极大地减少了内存读写，速度反而变快了。这就是“Flash (闪电)”的由来。
---


## 五、 Transformer 整体架构

Transformer 是由多个相同的 Block 堆叠而成的架构。一个标准的 Transformer Block 包含以下几个关键子模块：

### 1. 核心架构组件
*   **Multi-head Self-Attention**：负责词与词之间的全局信息交互。
*   **残差连接 (Residual Connection)**：即将输入直接加到输出上 ($x + f(x)$)，缓解梯度消失，有助于构建深层网络。
*   **层归一化 (Layer Normalization)**：
    *   在**特征维度**上对每个 Token 独立进行归一化。
    *   不同于 BatchNorm，LayerNorm 不受 Batch size 影响，不跨 Token 计算。
    *   作用：防止深层残差网络中的激活值爆炸或漂移，稳定注意力中的 Softmax。
*   **前馈神经网络 (FFN / MLP)**：
    *   通常是一个两层的 MLP，维度变化通常是 $D \rightarrow 4D \rightarrow D$。
    *   作用：Self-attention 只做信息的线性路由和混合，FFN 负责在单个 Token 内部进行**非线性特征变换**，极大增加模型容量。

### 2. 位置编码 (Positional Encoding)
*   因为自注意力机制没有位置概念，必须人为注入位置信息。
*   **现代解决方案 (RoPE)**：旋转位置编码 (Rotary Positional Embedding)，将位置映射为角度，通过旋转 Q 和 K 来编码**相对位置**。


## 六、 现代 Transformer 的架构微调 (Tweaking)

自 2017 年以来，Transformer 的宏观架构未发生根本改变，但在细节上衍生出了工业界常用的标准变体：

1.  **Pre-Norm (前置归一化)**：
    *   将 LayerNorm 移到残差连接**内部**（即在 Self-Attention 和 MLP 之前）。
    *   优势：使深层网络的训练更加稳定，避免了早期模型难以学习恒等映射的问题。
2.  **QK-Norm**：
    *   在计算注意力相似度前，对 Query 和 Key 进行归一化（通常使用 RMSNorm）。
    *   优势：防止梯度尖峰，进一步稳定训练。
3.  **SwiGLU MLP**：
    *   替换了传统的 ReLU/GELU 激活的 MLP。使用门控线性单元 (GLU) 的变体。
    *   计算方式类似于：$Y = (\sigma(XW_1) \odot XW_2)W_3$。
4.  **混合专家模型 (Mixture of Experts, MoE)**：
    *   在每个 Block 中学习 $E$ 个独立的 MLP（称为专家）。
    *   引入路由机制 (Router)，对于每个 Token，只激活 $A$ 个专家（$A < E$）。
    *   优势：**大幅增加模型参数量，但只带来适度的计算成本(FLOPs)增加**。
    *   问题：任务分配是困难点，很有可能只有几个expert干活，其他都不干活，导致参数浪费。

## 七、 视觉 Transformer (Vision Transformers, ViT)

Transformer 在自然语言处理取得成功后，被引入到计算机视觉领域。

### 1. ViT 的处理流程
1.  **Patch 切分**：将输入图像（如 $224 \times 224$）切分为不重叠的 Patches（如 $16 \times 16$）。
2.  **线性投影 (Flatten & Linear Projection)**：将每个 Patch 展平，并通过一个线性层映射为 $D$ 维向量（可以看作是步长为 16 的 $16 \times 16$ 卷积）。
3.  **加入位置编码**：为每个 Patch 加入 2D 位置编码。
4.  **Transformer 处理**：输入标准的 Transformer Block，不使用任何 Mask（每个 Patch 都能看到整张图像的所有 Patch）。
5.  **分类头 (Pooling / Class Token)**：对输出的向量序列进行平均池化（或使用特殊的 CLS token），接线性层进行分类。

### 2. ViT 与 CNN 的归纳偏置 (Inductive Biases) 对比
*   **CNN 的归纳偏置（强）**：
    *   *局部连通性 (Local connectivity)*：假设相邻像素是相关的。
    *   *平移等变性 (Translation equivariance)*：同一个卷积核在所有位置共享权值，特征随位置平移。
    *   *层次化 (Hierarchical)*：通过感受野逐步增大，自然构建“边缘->纹理->部件->物体”的层次。
*   **ViT 的归纳偏置（弱）**：
    *   除 Patch 结构和位置编码外，几乎没有硬编码的先验假设。
    *   全连接式的全局交互，从第一层开始就能建立任何 Patch 之间的长距离依赖。

### 3. 数据效率与扩展性规律
*   **小数据场景**：CNN 表现更好。CNN 凭借强归纳偏置，可以用更少的样本学习，不容易过拟合。早期 ViT 在小数据集（如 ImageNet-1K）上从头训练表现不佳。
*   **超大规模数据场景 (Scale)**：ViT 表现碾压 CNN。当数据量极其庞大（如 JFT-300M, 3亿图像）时，CNN 的强归纳偏置变成了限制上限的约束；而 ViT 凭借极弱的偏置和灵活的全局特征组织能力，展现出卓越的扩展性 (Scalability)。